In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog_name", "","")
catalog_name=dbutils.widgets.get("catalog_name")
vs_endpoint="test"

In [0]:
from databricks.vector_search.client import VectorSearchClient

In [0]:
# The following line automatically generates a PAT Token for authentication
client = VectorSearchClient()

# The following line uses the service principal token for authentication
# client = VectorSearchClient(service_principal_client_id=<CLIENT_ID>,service_principal_client_secret=<CLIENT_SECRET>)
is_already_exist=False
for i in client.list_endpoints()['endpoints']:
    if (i['name']==vs_endpoint):
        print("already exist")
        is_already_exist=True

        break

if(not is_already_exist):
   client.create_endpoint(
       name=vs_endpoint,
       endpoint_type="STANDARD" # or "STORAGE_OPTIMIZED"
   )

In [0]:
client = VectorSearchClient()
try:
   index = client.create_delta_sync_index(
     endpoint_name=vs_endpoint,
     source_table_name=f"{catalog_name}.gold.resume_data",
     index_name=f"{catalog_name}.gold.resume_data_index",
     pipeline_type="TRIGGERED",
     primary_key="resume_id",
     embedding_source_column="merged_description",
     embedding_model_endpoint_name="databricks-qwen3-embedding-0-6b", # This model is used for    ingestion, and is also used for querying unless model_endpoint_name_for_query is specified.
     
   )
except Exception as e:
    print(e)
    index.sync()

In [0]:
company_jd_1.  --- resume_id1_1,resume_id_2,resume_id_3,resume_id_4

company_jd_2 -- resume_id1_1,resume_id_2,resume_id_3,resume_id_4

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, FloatType, BooleanType, IntegerType, DoubleType,DateType
import pandas as pd
from pyspark.sql.functions import *

result_schema = ArrayType(StructType([
    
    StructField("Merged_Description", StringType()),
    StructField("resume_Id", IntegerType()),
    StructField("score", FloatType())
]))
from datetime import datetime
client = VectorSearchClient()
index = client.get_index(index_name=f'{catalog_name}.gold.resume_data_index')
@pandas_udf(result_schema)
def vector_search_results_udf(descriptions: pd.Series) -> pd.Series:
   

    def get_results(text):
        try:
            results = index.similarity_search(
                query_text=text,
                columns=["merged_description","resume_id"],
                num_results=3
            )
            formatted = []
            for r in results["result"]["data_array"]:
                # Adjust unpacking depending on index schema
              
                formatted.append({
                    "Merged_Description": r[0],
                    "resume_Id":r[1],
                    "score": float(r[2])
                })
            return formatted
        except Exception as e:
            print(f"Error during vector search for input: {str(text)[:30]}... → {e}")
            return []

    return descriptions.apply(get_results)

input_df = spark.read.table(f"{catalog_name}.gold.company_jd")

# Call the UDF
df_results = input_df.withColumn(
    "vs_matches",
    vector_search_results_udf(col("merged_description"))
)


In [0]:
df_results=df_results.select("company_jd_id","merged_description","vs_matches")

In [0]:
is_resume_company_data=spark.catalog.tableExists(f'{catalog_name}.gold.resume_company_data')

In [0]:
from delta.tables import DeltaTable
if(not is_resume_company_data):
    df_results.write.mode("overwrite").saveAsTable(f"{catalog_name}.gold.resume_company_data")
else:
    delta_target = DeltaTable.forName(spark, f'{catalog_name}.gold.resume_company_data')
    (
        delta_target.alias("t")
        .merge(
            df_results.alias("s"),
            "t.company_jd_id = s.company_jd_id"  # 👈 merge key, adjust as needed
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"✅ MERGE completed")


In [0]:
%sql
select * from resume_batch3.gold.resume_company_data

In [0]:
schema = ArrayType(
    StructType([
        StructField("Resume_Id", IntegerType(), True),
        StructField("Match", StringType(), True),
        StructField("Reason", StringType(), True)
    ])
)

In [0]:
%sql
select * from resume_batch3.gold.resume_company_data

In [0]:
df_final = spark.sql(fr"""
SELECT *
FROM (
  SELECT 
    company_jd_id
,
    from_json(
      ai_query(
        "databricks-meta-llama-3-3-70b-instruct",
        CONCAT(
           "You are an AI assistant that validates company jd with resume_data matches",
          "Check whether candidate resume is a good fit for the company jd and decide if it is a valid match. Give proper reason and dont include score in the reason",
          "Resume Data details: ",
          "company jd data = ", CAST(merged_description AS STRING), ", ",
          "resume_data matches (JSON list): ", to_json(vs_matches), ". ",
          "Respond only in valid JSON with fields: Match (Rejected/Selected), Reason (string) and Resume ID (int).Dont include json word"
        )
      ),
      'STRUCT<Match STRING, Reason STRING, Resume_ID int>'
    ) AS parsed_result
  FROM {catalog_name}.gold.resume_company_data
)
""")

In [0]:
df_final = spark.sql(fr"""
SELECT *
FROM (
  SELECT 
    company_jd_id
,
    from_json(
      ai_query(
        "databricks-meta-llama-3-3-70b-instruct",
        CONCAT(
           "You are an AI assistant that validates company jd with resume_data matches",
          "Check whether candidate resume is a good fit for the company jd and decide if it is a valid match. Give proper reason and dont include score in the reason",
          "Resume Data details: ",
          "company jd data = ", CAST(merged_description AS STRING), ", ",
          "resume_data matches (JSON list): ", to_json(vs_matches), ". ",
          "Respond only in valid JSON with fields: Match (Rejected/Selected), Reason (string) and Resume_ID (int).Dont include json word"
        )
      ),
      'STRUCT<Match STRING, Reason STRING, Resume_ID int>'
    ) AS parsed_result
  FROM {catalog_name}.gold.resume_company_data
)
""")

In [0]:
df_final.createOrReplaceTempView("temp_view")

In [0]:
schema = ArrayType(
    StructType([
        StructField("Resume_Id", IntegerType(), True),
        StructField("Match", StringType(), True),
        StructField("Reason", StringType(), True)
    ])
)

In [0]:
df_exploded = (
    df_final
    .select(
        "*",
        col("parsed_result.Resume_Id").alias("Resume_Id"),
        col("parsed_result.Match").alias("Match"),
        col("parsed_result.Reason").alias("Reason")
    )
    .drop("parsed_array", "resume")
).drop('parsed_result')

display(df_exploded)


In [0]:
is_resume_company_final_data=spark.catalog.tableExists(f'{catalog_name}.gold.resume_company_data_final')

In [0]:
from delta.tables import DeltaTable
if(not is_resume_company_final_data):
    df_results.write.mode("overwrite").saveAsTable(f"{catalog_name}.gold.resume_company_data_final")
else:
    delta_target = DeltaTable.forName(spark, f'{catalog_name}.gold.resume_company_data_final')
    (
        delta_target.alias("t")
        .merge(
            df_results.alias("s"),
            "t.company_jd_id = s.company_jd_id and t.Resume_Id = s.Resume_Id"  # 👈 merge key, adjust as needed
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"✅ MERGE completed")
